# 🎯 Reranking for Better RAG

**Boost retrieval accuracy with two-stage ranking**

---

## 📋 Overview

**What you'll learn:**
- Why reranking improves RAG
- Cross-encoder models
- Two-stage retrieval pipeline
- Production reranking strategies
- Cost vs quality tradeoffs

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
from typing import List, Dict, Tuple
import time

print("✅ Setup complete")

## 🤔 Why Reranking?

### Problem with Single-Stage Retrieval:
```
Query: "How do I install Python packages?"

Bi-encoder (fast but approximate):
  1. "Python package manager pip" ✅ (relevant)
  2. "Installing software on Linux" ❓ (somewhat relevant)
  3. "Package distribution systems" ❓
  4. "pip install command usage" ✅ (very relevant but ranked low!)
```

### Solution: Two-Stage Retrieval
```
Stage 1 (Bi-encoder): Fast, retrieve top 100
  ↓
Stage 2 (Cross-encoder): Accurate, rerank to top 5
  ↓
Better ranking!
```

### Benefits:
- 🎯 **Better ranking**: More accurate relevance
- ⚡ **Still fast**: Rerank only top candidates
- 📈 **Higher precision**: Best results in top K
- 💰 **Cost effective**: Rerank < Embed all

## 🔄 Bi-encoder vs Cross-encoder

### Bi-encoder (Stage 1):
```python
# Encode separately
query_emb = encode(query)      # [384]
doc_embs = encode(docs)         # [N, 384]
scores = cosine(query_emb, doc_embs)  # Fast!
```
✅ **Fast** (can precompute doc embeddings)
❌ **Less accurate** (no query-doc interaction)

### Cross-encoder (Stage 2):
```python
# Encode together
for doc in docs:
    score = model(query + doc)  # Direct relevance score
```
✅ **More accurate** (models query-doc interaction)
❌ **Slower** (can't precompute, need query)

## 🏗️ Basic Reranking Pipeline

In [ ]:
# Sample documents
documents = [
    "Python uses pip for package management. Install packages with 'pip install package_name'.",
    "Package managers help install software. Different systems use different managers.",
    "The Python Package Index (PyPI) hosts thousands of Python packages.",
    "To install a Python package, open terminal and run pip install.",
    "Linux systems have package managers like apt and yum.",
    "Virtual environments isolate Python package installations.",
    "You can upgrade Python packages using pip install --upgrade.",
    "Requirements.txt files list all project dependencies.",
]

class RerankerRAG:
    """RAG with two-stage retrieval and reranking."""
    
    def __init__(
        self,
        documents: List[str],
        bi_encoder_name: str = 'all-MiniLM-L6-v2',
        cross_encoder_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
    ):
        self.documents = documents
        
        # Stage 1: Bi-encoder for fast retrieval
        print(f"Loading bi-encoder: {bi_encoder_name}")
        self.bi_encoder = SentenceTransformer(bi_encoder_name)
        self.doc_embeddings = self.bi_encoder.encode(documents)
        
        # Stage 2: Cross-encoder for accurate reranking
        print(f"Loading cross-encoder: {cross_encoder_name}")
        self.cross_encoder = CrossEncoder(cross_encoder_name)
        
        print("✅ Reranker ready")
    
    def retrieve_stage1(self, query: str, top_k: int = 20) -> List[Dict]:
        """Stage 1: Fast bi-encoder retrieval."""
        query_emb = self.bi_encoder.encode([query])[0]
        
        # Cosine similarity
        similarities = np.dot(self.doc_embeddings, query_emb) / (
            np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
        )
        
        # Get top k
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'doc_id': int(idx),
                'document': self.documents[idx],
                'stage1_score': float(similarities[idx])
            })
        
        return results
    
    def rerank_stage2(self, query: str, candidates: List[Dict], top_k: int = 5) -> List[Dict]:
        """Stage 2: Cross-encoder reranking."""
        # Prepare pairs for cross-encoder
        pairs = [[query, cand['document']] for cand in candidates]
        
        # Get cross-encoder scores
        ce_scores = self.cross_encoder.predict(pairs)
        
        # Add scores to candidates
        for cand, score in zip(candidates, ce_scores):
            cand['stage2_score'] = float(score)
        
        # Sort by stage 2 score
        reranked = sorted(candidates, key=lambda x: x['stage2_score'], reverse=True)
        
        return reranked[:top_k]
    
    def search(
        self,
        query: str,
        stage1_k: int = 20,
        final_k: int = 5
    ) -> List[Dict]:
        """Two-stage search with reranking."""
        # Stage 1: Fast retrieval
        candidates = self.retrieve_stage1(query, top_k=stage1_k)
        
        # Stage 2: Accurate reranking
        reranked = self.rerank_stage2(query, candidates, top_k=final_k)
        
        return reranked

# Create reranker
reranker = RerankerRAG(documents)

In [ ]:
# Test query
query = "How do I install Python packages?"

print(f"🔍 Query: '{query}'\n")
print("="*80)

# Stage 1 only
print("\n📌 Stage 1 (Bi-encoder) - Top 5:\n")
stage1_results = reranker.retrieve_stage1(query, top_k=5)
for i, result in enumerate(stage1_results, 1):
    print(f"{i}. (score: {result['stage1_score']:.3f})")
    print(f"   {result['document'][:70]}...\n")

# Full two-stage
print("\n🎯 After Stage 2 (Reranked) - Top 5:\n")
final_results = reranker.search(query, stage1_k=20, final_k=5)
for i, result in enumerate(final_results, 1):
    print(f"{i}. (stage2: {result['stage2_score']:.3f}, stage1: {result['stage1_score']:.3f})")
    print(f"   {result['document'][:70]}...\n")

print("💡 Notice how reranking changed the order!")

## ⚡ Performance Comparison

In [ ]:
def benchmark_retrieval(reranker: RerankerRAG, query: str, n_runs: int = 5):
    """Benchmark retrieval speed."""
    
    # Bi-encoder only
    times_bi = []
    for _ in range(n_runs):
        start = time.time()
        _ = reranker.retrieve_stage1(query, top_k=5)
        times_bi.append(time.time() - start)
    
    # Two-stage
    times_rerank = []
    for _ in range(n_runs):
        start = time.time()
        _ = reranker.search(query, stage1_k=20, final_k=5)
        times_rerank.append(time.time() - start)
    
    return {
        'bi_encoder_only': np.mean(times_bi) * 1000,  # ms
        'with_reranking': np.mean(times_rerank) * 1000,  # ms
        'overhead': (np.mean(times_rerank) - np.mean(times_bi)) * 1000  # ms
    }

# Benchmark
print("⚡ Performance Benchmark\n")
results = benchmark_retrieval(reranker, query)

print(f"Bi-encoder only:  {results['bi_encoder_only']:.1f}ms")
print(f"With reranking:   {results['with_reranking']:.1f}ms")
print(f"Reranking overhead: {results['overhead']:.1f}ms")
print(f"\n💡 Reranking adds {results['overhead']:.0f}ms but improves accuracy significantly")

## 🎛️ Production Strategies

In [ ]:
class ProductionReranker:
    """Production-ready reranking with options."""
    
    def __init__(self, documents: List[str]):
        self.documents = documents
        self.bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_embeddings = self.bi_encoder.encode(documents)
        self.cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    
    def search(
        self,
        query: str,
        strategy: str = 'balanced',
        final_k: int = 5
    ) -> List[Dict]:
        """
        Search with different strategies.
        
        Strategies:
        - 'fast': No reranking (stage 1 only)
        - 'balanced': Rerank top 20 candidates
        - 'accurate': Rerank top 50 candidates
        """
        
        if strategy == 'fast':
            # Stage 1 only
            return self._retrieve_stage1(query, top_k=final_k)
        
        elif strategy == 'balanced':
            # Rerank top 20
            candidates = self._retrieve_stage1(query, top_k=20)
            return self._rerank_stage2(query, candidates, top_k=final_k)
        
        elif strategy == 'accurate':
            # Rerank top 50
            candidates = self._retrieve_stage1(query, top_k=50)
            return self._rerank_stage2(query, candidates, top_k=final_k)
        
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
    
    def _retrieve_stage1(self, query: str, top_k: int) -> List[Dict]:
        """Stage 1 retrieval."""
        query_emb = self.bi_encoder.encode([query])[0]
        similarities = np.dot(self.doc_embeddings, query_emb) / (
            np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(query_emb)
        )
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        return [{
            'doc_id': int(idx),
            'document': self.documents[idx],
            'score': float(similarities[idx])
        } for idx in top_indices]
    
    def _rerank_stage2(self, query: str, candidates: List[Dict], top_k: int) -> List[Dict]:
        """Stage 2 reranking."""
        pairs = [[query, cand['document']] for cand in candidates]
        ce_scores = self.cross_encoder.predict(pairs)
        
        for cand, score in zip(candidates, ce_scores):
            cand['rerank_score'] = float(score)
        
        reranked = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
        return reranked[:top_k]

# Test strategies
prod_reranker = ProductionReranker(documents)

print("🎛️ Testing Different Strategies\n")
print("="*80)

for strategy in ['fast', 'balanced', 'accurate']:
    print(f"\n📌 Strategy: {strategy}")
    
    start = time.time()
    results = prod_reranker.search(query, strategy=strategy, final_k=3)
    elapsed = (time.time() - start) * 1000
    
    print(f"   Latency: {elapsed:.1f}ms")
    print(f"   Top result: {results[0]['document'][:60]}...")

## 📊 Quality Improvement Analysis

In [ ]:
# Create test cases with ground truth
test_cases = [
    {
        "query": "How to install Python packages?",
        "relevant_ids": [0, 3, 6]  # Docs about pip install
    },
    {
        "query": "What is PyPI?",
        "relevant_ids": [2, 7]  # Docs about PyPI/packages
    },
    {
        "query": "How to upgrade packages?",
        "relevant_ids": [6]  # Doc about pip upgrade
    },
]

def evaluate_retrieval(retriever, test_cases, use_reranking=True):
    """Evaluate with/without reranking."""
    recalls = []
    
    for test in test_cases:
        if use_reranking:
            results = retriever.search(test['query'], strategy='balanced', final_k=5)
        else:
            results = retriever.search(test['query'], strategy='fast', final_k=5)
        
        retrieved_ids = [r['doc_id'] for r in results]
        relevant_set = set(test['relevant_ids'])
        retrieved_set = set(retrieved_ids)
        
        recall = len(relevant_set & retrieved_set) / len(relevant_set)
        recalls.append(recall)
    
    return np.mean(recalls)

# Compare
print("📊 Quality Comparison\n")

recall_without = evaluate_retrieval(prod_reranker, test_cases, use_reranking=False)
recall_with = evaluate_retrieval(prod_reranker, test_cases, use_reranking=True)

print(f"Recall@5 without reranking: {recall_without:.3f}")
print(f"Recall@5 with reranking:    {recall_with:.3f}")
print(f"\n📈 Improvement: {(recall_with - recall_without) * 100:.1f}%")

## ✅ Summary

### Two-Stage Retrieval:

```
Stage 1: Bi-Encoder (Fast)
├─ Retrieve top 20-50 candidates
├─ ~10-50ms
└─ Precomputed embeddings

Stage 2: Cross-Encoder (Accurate)
├─ Rerank to top 5
├─ +50-200ms
└─ Better relevance
```

### When to Rerank:

✅ **Use reranking when:**
- Accuracy is critical
- Can afford 50-200ms latency
- Have complex queries
- Precision matters

❌ **Skip reranking when:**
- Need <100ms latency
- Simple keyword queries
- Limited compute budget
- Bi-encoder works well enough

### Model Selection:

**Bi-Encoders (Stage 1):**
- `all-MiniLM-L6-v2`: Fast, general
- `all-mpnet-base-v2`: Better quality
- `e5-large-v2`: Best quality

**Cross-Encoders (Stage 2):**
- `ms-marco-MiniLM-L-6-v2`: Fast reranking
- `ms-marco-MiniLM-L-12-v2`: Better accuracy
- `ms-marco-electra-base`: Highest quality

### Cost-Benefit Analysis:

| Metric | Without Rerank | With Rerank | Change |
|--------|----------------|-------------|--------|
| Latency | 20ms | 120ms | +100ms |
| Recall@5 | 0.70 | 0.85 | +15% |
| Cost/1K | $0.10 | $0.30 | +$0.20 |
| User Sat | 70% | 90% | +20% |

**ROI**: Worth it if better results justify cost

### Production Strategies:

```python
# Strategy 1: Always rerank
results = search(query, stage1_k=20, final_k=5)

# Strategy 2: Adaptive
if query_complexity > threshold:
    results = search(query, strategy='accurate')
else:
    results = search(query, strategy='fast')

# Strategy 3: A/B test
if user_id % 2 == 0:
    results = search_with_rerank(query)
else:
    results = search_without_rerank(query)
```

### Optimization Tips:

1. **Batch reranking**
   ```python
   # Rerank all candidates at once
   scores = cross_encoder.predict(all_pairs)
   ```

2. **Cache stage 1**
   ```python
   # Cache bi-encoder results for common queries
   ```

3. **Right stage 1 size**
   ```python
   # Too small: Miss relevant docs
   # Too large: Slow reranking
   # Sweet spot: 20-50 candidates
   ```

4. **Monitor latency**
   ```python
   # Set timeout for stage 2
   # Fallback to stage 1 if timeout
   ```

### Typical Results:

```
Bi-encoder only:
  Recall@5: 0.70
  Latency:  20ms

With reranking:
  Recall@5: 0.85  (+21%)
  Latency:  120ms (+100ms)
  
Worth it? Usually yes for production RAG!
```

### Next: `05_rag_systems/07_citations.ipynb`